# Wildfire Prediction — Ensemble Model
Random Forest + Extra Trees + Gradient Boosting + Logistic Regression + Cox PH Survival

In [43]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               ExtraTreesClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize

In [44]:
train = pd.read_csv("content/FE_train.csv")
test  = pd.read_csv("content/FE_test.csv")

with open("feature_selection/final_feature_set.json") as f:
    selected_features = json.load(f)["final_features"]

print(f"Train: {train.shape}  |  Test: {test.shape}")
print(f"Selected features: {len(selected_features)}")

Train: (221, 47)  |  Test: (95, 45)
Selected features: 30


In [45]:
def add_features(df):
    df = df.copy()
    # Interaction: fire growth x spread speed
    df["growth_x_speed"]    = df["radial_growth_m"] * df["centroid_speed_m_per_h"]
    # How far the fire is relative to its closing speed (estimated time-to-breach)
    df["dist_over_closing"] = df["dist_min_ci_0_5h"] / (df["closing_speed_abs_m_per_h"] + 1e-6)
    # Growth quality weighted by trajectory fit quality
    df["growth_x_fit"]      = df["area_growth_rel_0_5h"] * df["dist_fit_r2_0_5h"]
    # Bearing alignment x speed (how directly fire heads toward zone)
    df["aligned_speed"]     = df["alignment_abs"] * df["centroid_speed_m_per_h"]
    # Rate of change of distance vs current distance
    df["dist_trend_ratio"]  = df["dist_slope_ci_0_5h"] / (df["dist_min_ci_0_5h"] + 1e-6)
    # Log of estimated breach time
    df["log_time_to_close"] = np.log1p(df["dist_over_closing"].clip(0))
    return df

X_train = add_features(train[selected_features])
X_test  = add_features(test[selected_features])
print(f"Total features after engineering: {X_train.shape[1]}")

Total features after engineering: 36


In [46]:
# Create binary label for each time horizon:
# label=1 if fire HIT within H hours, 0 otherwise (censored OR hit later)
horizons = [12, 24, 48, 72]

for H in horizons:
    train[f"hit_{H}h"] = ((train["event"] == 1) & (train["time_to_hit_hours"] <= H)).astype(int)

print("Label distributions:")
for H in horizons:
    print(f"  {H}h: {train[f'hit_{H}h'].mean():.3f} positive rate  (n={train[f'hit_{H}h'].sum()})")

Label distributions:
  12h: 0.222 positive rate  (n=49)
  24h: 0.285 positive rate  (n=63)
  48h: 0.299 positive rate  (n=66)
  72h: 0.312 positive rate  (n=69)


In [47]:
class CoxPHSurvival:
    """
    Cox Proportional Hazards model with Breslow baseline estimator.
    Implemented using scipy L-BFGS-B (no extra packages required).
    alpha: L2 regularisation strength
    """
    def __init__(self, alpha=0.5):
        self.alpha  = alpha
        self.coef_  = None
        self.scaler = StandardScaler()

    def _partial_loglik(self, beta, X, time, event):
        """Negative partial log-likelihood + L2 penalty."""
        Xb    = X @ beta
        order = np.argsort(-time)
        Xb_o  = Xb[order]
        ev_o  = event[order]
        log_lik = 0.0
        log_cum = np.logaddexp.accumulate(Xb_o)
        for i in range(len(Xb_o)):
            if ev_o[i]:
                log_lik += Xb_o[i] - log_cum[i]
        return -(log_lik - 0.5 * self.alpha * np.dot(beta, beta))

    def _grad(self, beta, X, time, event):
        """Gradient of negative partial log-likelihood."""
        Xb    = X @ beta
        order = np.argsort(-time)
        X_o, Xb_o, ev_o = X[order], Xb[order], event[order]
        grad = np.zeros_like(beta)
        log_sum, sum_expX = -np.inf, np.zeros(X.shape[1])
        for i in range(len(Xb_o)):
            log_sum  = np.logaddexp(log_sum, Xb_o[i])
            sum_expX += np.exp(Xb_o[i]) * X_o[i]
            if ev_o[i]:
                grad += sum_expX / np.exp(log_sum) - X_o[i]
        return grad + self.alpha * beta

    def fit(self, X, time, event):
        Xs = self.scaler.fit_transform(X)
        res = minimize(self._partial_loglik, np.zeros(Xs.shape[1]),
                       args=(Xs, time, event), jac=self._grad,
                       method="L-BFGS-B", options={"maxiter": 500})
        self.coef_ = res.x
        self._fit_baseline(Xs, time, event)
        return self

    def _fit_baseline(self, Xs, time, event):
        """Breslow estimator for cumulative baseline hazard H0(t)."""
        Xb    = Xs @ self.coef_
        order = np.argsort(time)
        t_o, ev_o, Xb_o = time[order], event[order], Xb[order]
        times, h0 = [], []
        for i in range(len(t_o)):
            if ev_o[i]:
                times.append(t_o[i])
                h0.append(1.0 / (np.sum(np.exp(Xb_o[i:])) + 1e-12))
        self.baseline_times_ = np.array(times)
        self.baseline_H0_    = np.cumsum(h0)

    def predict_hit_proba(self, X, t):
        """P(T <= t | X) for each sample."""
        Xs   = self.scaler.transform(X)
        Xb   = Xs @ self.coef_
        idx  = np.searchsorted(self.baseline_times_, t, side='right') - 1
        H0_t = self.baseline_H0_[idx] if idx >= 0 else 0.0
        surv = np.exp(-H0_t * np.exp(Xb))
        return 1.0 - surv

    def risk_score(self, X):
        """Higher score = higher risk = expected to hit sooner."""
        return np.exp(self.scaler.transform(X) @ self.coef_)

print("CoxPHSurvival class defined.")


CoxPHSurvival class defined.


In [48]:
print("Fitting Cox PH model (uses full survival structure)...")

cox = CoxPHSurvival(alpha=0.5)
cox.fit(X_train.values, train["time_to_hit_hours"].values, train["event"].values)
print("Cox PH fitted.")

max_time = train["time_to_hit_hours"].max()  # 66.99h — cap for 72h horizon

cox_preds = {}
for H in horizons:
    t = min(H, max_time)
    cox_preds[H] = cox.predict_hit_proba(X_test.values, t)
    print(f"  Cox P(hit<={H}h): mean={cox_preds[H].mean():.3f}")

Fitting Cox PH model (uses full survival structure)...


Cox PH fitted.
  Cox P(hit<=12h): mean=0.206
  Cox P(hit<=24h): mean=0.281
  Cox P(hit<=48h): mean=0.303
  Cox P(hit<=72h): mean=0.420


In [49]:
def build_ensemble(X, y, X_test_data, seed=42):
    """Train RF + ET + GB + LR ensemble for one horizon label."""
    models = [
        ("RF", RandomForestClassifier(
            n_estimators=500, min_samples_leaf=2, min_samples_split=5,
            max_features="sqrt", class_weight="balanced",
            random_state=seed, n_jobs=-1)),
        ("ET", ExtraTreesClassifier(
            n_estimators=500, min_samples_leaf=2, min_samples_split=5,
            max_features="sqrt", class_weight="balanced",
            random_state=seed+1, n_jobs=-1)),
        ("GB", GradientBoostingClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=4,
            min_samples_leaf=3, subsample=0.8,
            max_features="sqrt", random_state=seed+2)),
        ("LR", Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    LogisticRegression(C=0.1, class_weight="balanced",
                                           max_iter=1000, random_state=seed+3))])),
    ]
    skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    preds = []
    for name, model in models:
        oof = cross_val_predict(model, X, y, cv=skf, method="predict_proba")[:, 1]
        print(f"    {name}: OOF AUC = {roc_auc_score(y, oof):.4f}")
        model.fit(X, y)
        preds.append(model.predict_proba(X_test_data)[:, 1])
    return np.mean(preds, axis=0)

print("Ensemble function defined.")

Ensemble function defined.


In [50]:
ensemble_preds = {}
for H in horizons:
    print(f"--- Horizon {H}h ---")
    ensemble_preds[H] = build_ensemble(
        X_train.values,
        train[f"hit_{H}h"].values,
        X_test.values
    )

--- Horizon 12h ---
    RF: OOF AUC = 0.9486
    ET: OOF AUC = 0.9229
    GB: OOF AUC = 0.9413
    LR: OOF AUC = 0.9067
--- Horizon 24h ---
    RF: OOF AUC = 0.9899
    ET: OOF AUC = 0.9700
    GB: OOF AUC = 0.9856
    LR: OOF AUC = 0.9242
--- Horizon 48h ---
    RF: OOF AUC = 0.9970
    ET: OOF AUC = 0.9732
    GB: OOF AUC = 0.9940
    LR: OOF AUC = 0.9082
--- Horizon 72h ---
    RF: OOF AUC = 1.0000
    ET: OOF AUC = 0.9782
    GB: OOF AUC = 0.9998
    LR: OOF AUC = 0.9205


In [51]:
# Blend: 30% Cox PH (survival structure) + 70% Ensemble (non-linear patterns)
COX_WEIGHT = 0.30

blended = {H: COX_WEIGHT * cox_preds[H] + (1 - COX_WEIGHT) * ensemble_preds[H]
           for H in horizons}

# Enforce monotonicity: P(12h) <= P(24h) <= P(48h) <= P(72h)
p12 = np.clip(blended[12], 0, 1)
p24 = np.clip(np.maximum(blended[24], p12), 0, 1)
p48 = np.clip(np.maximum(blended[48], p24), 0, 1)
p72 = np.clip(np.maximum(blended[72], p48), 0, 1)

print("Monotonicity check (all should be True):")
print(f"  P(12h) <= P(24h): {np.all(p24 >= p12 - 1e-9)}")
print(f"  P(24h) <= P(48h): {np.all(p48 >= p24 - 1e-9)}")
print(f"  P(48h) <= P(72h): {np.all(p72 >= p48 - 1e-9)}")

Monotonicity check (all should be True):
  P(12h) <= P(24h): True
  P(24h) <= P(48h): True
  P(48h) <= P(72h): True


In [53]:
from sklearn.model_selection import train_test_split

# ---- 1. Split ----
train_df, val_df = train_test_split(
    train, test_size=0.2, stratify=train["event"], random_state=42
)
X_tr  = add_features(train_df[selected_features])
X_val = add_features(val_df[selected_features])

print(f"Train split: {len(train_df)}  |  Val split: {len(val_df)}")

# ---- 2. Fit Cox on train split ----
cox_val = CoxPHSurvival(alpha=0.5)
cox_val.fit(X_tr.values, train_df["time_to_hit_hours"].values, train_df["event"].values)

# ---- 3. Fit ensemble on train split ----
def train_models(X, y, seed=42):
    """Fit all 4 models and return them."""
    models = [
        RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
            max_features="sqrt", class_weight="balanced", random_state=seed, n_jobs=-1),
        ExtraTreesClassifier(n_estimators=300, min_samples_leaf=2,
            max_features="sqrt", class_weight="balanced", random_state=seed+1, n_jobs=-1),
        GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
            max_depth=4, subsample=0.8, max_features="sqrt", random_state=seed+2),
        Pipeline([("s", StandardScaler()),
                  ("c", LogisticRegression(C=0.1, class_weight="balanced", max_iter=1000))])
    ]
    for m in models:
        m.fit(X, y)
    return models

val_ensemble_models = {}
for H in horizons:
    labels = ((train_df["event"] == 1) & (train_df["time_to_hit_hours"] <= H)).astype(int)
    val_ensemble_models[H] = train_models(X_tr.values, labels.values)
    print(f"  Trained ensemble for {H}h horizon")

# ---- 4. Blended val predictions ----
COX_WEIGHT    = 0.30
max_train_t   = train_df["time_to_hit_hours"].max()

blended_val = {}
for H in horizons:
    cox_p = cox_val.predict_hit_proba(X_val.values, min(H, max_train_t))
    ens_p = np.mean([m.predict_proba(X_val.values)[:, 1]
                     for m in val_ensemble_models[H]], axis=0)
    blended_val[H] = np.clip(COX_WEIGHT * cox_p + (1 - COX_WEIGHT) * ens_p, 0, 1)

# Enforce monotonicity
p12v = blended_val[12]
p24v = np.maximum(blended_val[24], p12v)
p48v = np.maximum(blended_val[48], p24v)
p72v = np.maximum(blended_val[72], p48v)
surv_val = {12: 1-p12v, 24: 1-p24v, 48: 1-p48v, 72: 1-p72v}

# ---- 5. Concordance index ----
def concordance_index(event_times, event_observed, risk_scores):
    concordant = discordant = tied = 0
    n = len(event_times)
    for i in range(n):
        for j in range(i+1, n):
            if event_times[i] == event_times[j]:
                continue
            if not event_observed[i] and not event_observed[j]:
                continue
            if event_times[i] < event_times[j]:
                if not event_observed[i]: continue
                e, l = i, j
            else:
                if not event_observed[j]: continue
                e, l = j, i
            if   risk_scores[e] > risk_scores[l]: concordant += 1
            elif risk_scores[e] < risk_scores[l]: discordant += 1
            else:                                  tied       += 1
    total = concordant + discordant + tied
    return (concordant + 0.5 * tied) / total if total > 0 else 0.5

val_times  = val_df["time_to_hit_hours"].values
val_events = val_df["event"].values
tr_times   = train_df["time_to_hit_hours"].values
tr_events  = train_df["event"].values

risk_val = cox_val.risk_score(X_val.values)
c_index  = concordance_index(val_times, val_events, risk_val)

# ---- 6. Brier score (IPCW) ----
def brier_score_ipcw(surv_val_probs, val_times, val_events,
                     tr_times, tr_events, t):
    """Inverse probability of censoring weighted Brier score at time t."""
    # Kaplan-Meier estimate of censoring distribution G(t)
    order  = np.argsort(tr_times)
    t_sort = tr_times[order]
    ev_sort = tr_events[order]
    km_t, km_g = [0.0], [1.0]
    n_risk = len(tr_times)
    for i in range(len(t_sort)):
        if ev_sort[i] == 0:   # censored observation
            km_g.append(km_g[-1] * (1 - 1 / n_risk))
            km_t.append(t_sort[i])
        n_risk -= 1
    km_t = np.array(km_t); km_g = np.array(km_g)

    def G(s):
        idx = np.searchsorted(km_t, s, side="right") - 1
        return km_g[max(idx, 0)]

    bs = 0.0
    n  = len(val_times)
    for i in range(n):
        S_t = surv_val_probs[i]
        Ti, di = val_times[i], val_events[i]
        if Ti <= t and di == 1:
            bs += (1 / (G(Ti) + 1e-12)) * (0 - S_t) ** 2
        elif Ti > t:
            bs += (1 / (G(t)  + 1e-12)) * (1 - S_t) ** 2
        # censored before t: excluded
    return bs / n

eval_horizons = [24, 48, min(72, max_train_t - 1e-6)]
brier_weights = np.array([0.3, 0.4, 0.3])
brier_scores  = []

print("Brier scores:")
for H in eval_horizons:
    H_key = min(horizons, key=lambda x: abs(x - H))
    bs = brier_score_ipcw(surv_val[H_key], val_times, val_events,
                          tr_times, tr_events, H)
    brier_scores.append(bs)
    print(f"  t={H:.0f}h : {bs:.4f}")

weighted_brier = np.sum(brier_weights * np.array(brier_scores))

# ---- 7. Hybrid score ----
hybrid = 0.3 * c_index + 0.7 * (1 - weighted_brier)

print("f{'='*40}")
print(f"C-index        : {c_index:.4f}")
print(f"Weighted Brier : {weighted_brier:.4f}")
print(f"Hybrid score   : {hybrid:.4f}  ({hybrid*100:.2f}%)")
print(f"{'='*40}")
print("Note: local val score uses only 20% of data — Kaggle score may differ.")

Train split: 176  |  Val split: 45
  Trained ensemble for 12h horizon
  Trained ensemble for 24h horizon
  Trained ensemble for 48h horizon
  Trained ensemble for 72h horizon
Brier scores:
  t=24h : 0.0622
  t=48h : 0.0301
  t=67h : 0.0053
f{'='*40}
C-index        : 0.8963
Weighted Brier : 0.0323
Hybrid score   : 0.9463  (94.63%)
Note: local val score uses only 20% of data — Kaggle score may differ.


In [ ]:
submission = pd.DataFrame({
    "event_id": test["event_id"],
    "prob_12h":  p12,
    "prob_24h":  p24,
    "prob_48h":  p48,
    "prob_72h":  p72
})

submission.to_csv("content/submission_final.csv", index=False)
print("Saved to content/submission_final.csv")
print(submission.head(10).to_string(index=False))

print("Stats:")
for col in ["prob_12h", "prob_24h", "prob_48h", "prob_72h"]:
    s = submission[col]
    print(f"  {col}: mean={s.mean():.3f}  std={s.std():.3f}  min={s.min():.3f}  max={s.max():.3f}")

Saved to content/submission_final.csv
 event_id  prob_12h  prob_24h  prob_48h  prob_72h
 10662602  0.040820  0.052054  0.052054  0.069984
 13353600  0.558586  0.762209  0.782691  0.863816
 13942327  0.121292  0.167973  0.177092  0.261422
 16112781  0.644971  0.758753  0.777364  0.855508
 17132808  0.685997  0.685997  0.685997  0.685997
 17445696  0.034809  0.037211  0.037710  0.038761
 17599982  0.018926  0.019075  0.019711  0.024337
 18750374  0.267950  0.303455  0.544887  0.595707
 21365245  0.067359  0.079465  0.083425  0.096963
 23634840  0.376953  0.768399  0.803463  0.887786
Stats:
  prob_12h: mean=0.241  std=0.268  min=0.010  max=0.909
  prob_24h: mean=0.312  std=0.329  min=0.010  max=0.936
  prob_48h: mean=0.325  std=0.339  min=0.010  max=0.941
  prob_72h: mean=0.366  std=0.362  min=0.010  max=0.948
